In [1]:
import json, os, glob
from datetime import datetime, timezone
from collections import defaultdict, Counter

# --- Configuration ---
BASE_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ3\mineV9.3"
# Treat <14d add→remove trials as excluded (True to match the paper; False to keep everything)
APPLY_14D_PERSISTENCE = True
PERSISTENCE_DAYS = 14

def parse_dt(s):
    # Examples: "2025-08-10 23:59:59 +0000" or ISO with timezone
    try:
        # ISO-ish with offset like 2020-12-20T17:43:54+01:00
        return datetime.fromisoformat(s.replace("Z", "+00:00"))
    except Exception:
        # Space-delimited with +0000 offset
        try:
            dt, off = s.rsplit(" ", 1)
            return datetime.strptime(dt, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)
        except Exception:
            # Fallback: date-only
            return datetime.fromisoformat(s).replace(tzinfo=timezone.utc)

def dt_days(a, b):
    return (b - a).total_seconds() / 86400.0

def load_timeline(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    # ensure chronological ordering by date
    tl = data.get("timeline", [])
    for e in tl:
        e["_dt"] = parse_dt(e["date"])
        e["_style_set"] = tuple(sorted(e.get("styles", [])))
    tl.sort(key=lambda x: x["_dt"])
    cutoff = parse_dt(data.get("cutoff_date")) if data.get("cutoff_date") else None
    return data.get("repo_name"), tl, cutoff

def build_episodes(tl, cutoff):
    """
    Episode = maximal span of consecutive timeline points with identical style_set.
    Each boundary occurs where style_set changes; last episode ends at next boundary or cutoff.
    """
    episodes = []
    if not tl:
        return episodes
    i = 0
    while i < len(tl):
        start = tl[i]
        styles = start["_style_set"]
        j = i + 1
        while j < len(tl) and tl[j]["_style_set"] == styles:
            j += 1
        end_dt = tl[j]["_dt"] if j < len(tl) else cutoff
        # exclude degenerate if no cutoff
        if end_dt is None:
            i = j
            continue
        episodes.append({
            "start_dt": start["_dt"],
            "end_dt": end_dt,
            "styles": styles,
            # within-episode changes are the interior points (excluding start)
            "within_changes": max(0, (j - i) - 1),
            "points": tl[i:j],  # raw points for debugging
        })
        i = j
    return episodes

def apply_persistence_filter(episodes):
    """
    If APPLY_14D_PERSISTENCE: drop episodes that are 'add→remove trials' <14 days.
    We approximate trials as episodes that end before the global cutoff (handled upstream)
    and whose duration < 14 days.
    """
    if not APPLY_14D_PERSISTENCE:
        return episodes
    kept = []
    for ep in episodes:
        dur = dt_days(ep["start_dt"], ep["end_dt"])
        # Keep episodes that either (a) lasted >=14d or (b) end at cutoff (unknown removal)
        if dur >= PERSISTENCE_DAYS or ep["end_dt"] == max(ep["end_dt"], ep["start_dt"], ep["end_dt"]):
            kept.append(ep)
        else:
            # drop <14d add→remove trials
            pass
    return kept

def per_style_metrics(episodes):
    out = []
    for ep in episodes:
        dur_days = dt_days(ep["start_dt"], ep["end_dt"])
        for s in ep["styles"]:
            # Count all within-episode changes toward each style present in the episode.
            changes = ep["within_changes"]
            rate = (changes / dur_days * 100.0) if dur_days > 0 else 0.0  # changes per 100 days
            out.append({
                "style": s,
                "start": ep["start_dt"].isoformat(),
                "end": ep["end_dt"].isoformat(),
                "duration_days": dur_days,
                "changes": changes,
                "changes_per_100d": rate,
            })
    return out

def main():
    paths = glob.glob(os.path.join(BASE_DIR, "**", "*.emulator_timeline.json"), recursive=True)
    all_episode_rows, all_style_rows = [], []
    for p in paths:
        try:
            repo, tl, cutoff = load_timeline(p)
            if not tl or not cutoff:
                continue
            episodes = build_episodes(tl, cutoff)
            episodes = apply_persistence_filter(episodes)
            # Add repo id
            for ep in episodes:
                ep["repo"] = repo
            all_episode_rows.extend(episodes)

            style_rows = per_style_metrics(episodes)
            for r in style_rows:
                r["repo"] = repo
            all_style_rows.extend(style_rows)
        except Exception as e:
            print("Failed on", p, e)

    # Aggregate per style across repos
    agg = defaultdict(lambda: {"episodes": 0, "total_changes": 0, "total_days": 0.0})
    for r in all_style_rows:
        a = agg[r["style"]]
        a["episodes"] += 1
        a["total_changes"] += r["changes"]
        a["total_days"] += r["duration_days"]

    print("Per-style aggregates:")
    for s, a in agg.items():
        rate = (a["total_changes"] / a["total_days"] * 100.0) if a["total_days"] > 0 else 0.0
        print(f"- {s}: episodes={a['episodes']}, changes={a['total_changes']}, "
              f"days={a['total_days']:.1f}, changes/100d={rate:.2f}")

    # Optional: write CSVs
    import csv
    with open("rq3_episodes.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["repo","styles","start_dt","end_dt","within_changes"])
        w.writeheader()
        for ep in all_episode_rows:
            w.writerow({
                "repo": ep["repo"],
                "styles": "+".join(ep["styles"]),
                "start_dt": ep["start_dt"].isoformat(),
                "end_dt": ep["end_dt"].isoformat(),
                "within_changes": ep["within_changes"],
            })
    with open("rq3_style_workload.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["repo","style","start","end","duration_days","changes","changes_per_100d"])
        w.writeheader()
        w.writerows(all_style_rows)

if __name__ == "__main__":
    main()


Per-style aggregates:
- Emu_Custom: episodes=197, changes=0, days=407914.5, changes/100d=0.00
- Emu_Community: episodes=281, changes=3, days=268060.1, changes/100d=0.00
- ThirdParty: episodes=35, changes=1, days=31270.4, changes/100d=0.00
- GMD: episodes=42, changes=0, days=15761.6, changes/100d=0.00
